# Third Model - HistGradientBoostingRegressor

**Why this model?** Ridge Regression provides a useful linear baseline, but its ability to capture complex relationships depends on manually engineered features.

The exploratory analysis revealed several patterns that may be difficult to represent with a purely additive linear model:

- The relationship between temperature and rental demand is non-linear.
- Hourly demand patterns differ substantially between working days and non-working days.
- Weather conditions may affect demand differently depending on the hour.
- Demand contains strong temporal patterns and interactions between calendar features.

Gradient Boosting offers a different approach. Instead of specifying these interactions manually, it builds an ensemble of decision trees sequentially, with each new tree correcting errors made by the previous ones.

This makes it possible to learn non-linear relationships and feature interactions automatically.

We use `HistGradientBoostingRegressor` from scikit-learn. It is computationally efficient for datasets of this size and supports missing values natively.

The main research question remains unchanged:

**How does the difference between random, blocked, and chronological validation depend on the model type?**

We compare two loss functions and examine whether a more flexible model improves predictions, particularly during peak-demand periods.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.inspection import permutation_importance

from src.data import load_data
from src.validation import RESULTS_DIR, SCHEMES, SEED, compute_metrics, evaluate, get_cv

In [2]:
X, y = load_data(drop__yr=True)

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())

X shape: (17379, 9)
y shape: (17379,)


,mnth,hr,holiday,weekday,workingday,weathersit,temp,hum,windspeed
datetime,,,,,,,,,
2011-01-01 00:00:00,1,0,0,6,0,1,0.24,0.81,0.0
2011-01-01 01:00:00,1,1,0,6,0,1,0.22,0.80,0.0
2011-01-01 02:00:00,1,2,0,6,0,1,0.22,0.80,0.0
2011-01-01 03:00:00,1,3,0,6,0,1,0.24,0.75,0.0
2011-01-01 04:00:00,1,4,0,6,0,1,0.24,0.75,0.0


In [3]:
assert X.index.equals(y.index)
assert X.index.is_monotonic_increasing
assert X.index.is_unique
assert "yr" not in X.columns
assert X["hum"].isna().sum() == 22

print("Chronological order:", X.index.is_monotonic_increasing)
print("Unique timestamps:", X.index.is_unique)
print("Year excluded:", "yr" not in X.columns)
print("Missing humidity values:", X["hum"].isna().sum())
print("Validation schemes:", SCHEMES)

Chronological order: True
Unique timestamps: True
Year excluded: True
Missing humidity values: 22
Validation schemes: ('random', 'blocked', 'chrono')


In [4]:
print("Features:")
print(X.columns.tolist())

print("\nMissing values:")
display(X.isna().sum())

Features:
['mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'hum', 'windspeed']

Missing values:


mnth           0
hr             0
holiday        0
weekday        0
workingday     0
weathersit     0
temp           0
hum           22
windspeed      0
dtype: int64

## Preprocessing

Unlike Ridge Regression, tree-based models do not require feature scaling or one-hot encoding.

We use the original numerical features returned by `load_data()`:

- Calendar features (`hr`, `mnth`, `weekday`)
- Binary indicators (`holiday`, `workingday`)
- Weather features (`weathersit`, `temp`, `hum`, `windspeed`)

The model learns split thresholds directly from these features. Non-linear relationships and interactions are captured by the tree structure rather than manually engineered transformations.

Missing humidity values are retained as `NaN`. `HistGradientBoostingRegressor` supports missing values natively and learns how to handle them during training.

No imputation, scaling, polynomial expansion, or manual interaction features are applied.

## Gradient Boosting and Hyperparameters

Gradient Boosting builds an ensemble of decision trees sequentially. Each new tree attempts to reduce the remaining prediction error of the current ensemble.

The complexity and learning behaviour of the model are controlled by several hyperparameters:

- `learning_rate` controls the contribution of each new tree. Smaller values usually require more iterations.
- `max_iter` determines the number of boosting iterations.
- `max_leaf_nodes` limits the number of leaves in each tree and therefore controls its complexity.
- `min_samples_leaf` specifies the minimum number of training observations in a leaf. Larger values produce smoother models and reduce the ability to fit very local patterns.
- `l2_regularization` penalises large leaf values and can help prevent overfitting.

We disable automatic early stopping (`early_stopping=False`). By default, early stopping may use a random validation subset of the training data. This would introduce an additional random validation procedure into our chronological experiment.

Instead, the number of iterations is selected through the same inner cross-validation strategy used for the corresponding outer validation scheme.

In [5]:
PARAM_GRID = {
    "learning_rate": [0.05, 0.1],
    "max_iter": [200, 500],
    "max_leaf_nodes": [15, 31],
    "min_samples_leaf": [20, 100],
    "l2_regularization": [0.0, 10.0],
}

In [6]:
def make_hgb_search(scheme, loss="squared_error"):
    model = HistGradientBoostingRegressor(loss=loss, early_stopping=False, random_state=SEED)

    inner_cv = get_cv(scheme, n_splits=3)

    search = GridSearchCV(
        estimator=model,
        param_grid=PARAM_GRID,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        refit=True,
        n_jobs=-1,
        return_train_score=True
    )

    return search

In [7]:
print("Target minimum:", y.min())
print("Target maximum:", y.max())
print("All target values non-negative:", (y >= 0).all())

Target minimum: 1
Target maximum: 977
All target values non-negative: True


## Computational Cost

Before running the full experiment, we measure the time required for one hyperparameter search.
The grid contains 32 parameter combinations, evaluated with three inner cross-validation splits. The complete experiment repeats this search across the outer folds, validation schemes, and loss functions.
This preliminary benchmark helps estimate the total computational cost.

In [10]:
benchmark_scheme = "random"

start = time.perf_counter()

benchmark_search = make_hgb_search(scheme=benchmark_scheme, loss="squared_error")
benchmark_search.fit(X, y)
benchmark_time = time.perf_counter() - start

print(f"Benchmark time: {benchmark_time:.2f} seconds")
print("Best parameters:", benchmark_search.best_params_)
print("Best inner CV RMSE:", -benchmark_search.best_score_)

Benchmark time: 67.86 seconds
Best parameters: {'l2_regularization': 0.0, 'learning_rate': 0.1, 'max_iter': 500, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
Best inner CV RMSE: 63.415362890235684


## Validation Strategy and Hyperparameter Tuning

We use the same nested cross-validation framework as in the Ridge experiment.

The outer validation estimates generalisation performance using three schemes:

- **Random K-Fold:** randomly distributes observations across training and test folds.
- **Blocked validation:** evaluates the model on contiguous time blocks while allowing training observations from both sides of the test block.
- **Chronological validation:** trains on earlier observations and evaluates on later observations.

For each outer training fold, `GridSearchCV` selects the hyperparameters using three inner splits generated by the corresponding validation scheme.

The outer test fold is not used during hyperparameter selection. After selecting the best parameter combination, the estimator is refitted on the full outer training fold and evaluated on the untouched test fold.

Automatic early stopping is disabled to avoid introducing a separate random validation subset into the chronological experiment.

## Model Training and Evaluation

We evaluate two HistGradientBoostingRegressor variants:

- **HGB:** uses the squared error loss.
- **HGB Poisson:** uses the Poisson loss with a logarithmic link function.

Both variants use the same feature set, hyperparameter grid, and nested cross-validation procedure.

We evaluate each model under all three validation schemes to determine how the model's flexibility affects the difference between random, blocked, and chronological validation.

In [11]:
hgb_results = {}

loss_variants = {"hgb": "squared_error", "hgb_poisson": "poisson",}

for name, loss in loss_variants.items():
    for scheme in SCHEMES:
        print(f"\nRunning {name} — {scheme}")

        search = make_hgb_search(scheme=scheme, loss=loss)
        start = time.perf_counter()

        folds, preds = evaluate(model=search, X=X, y=y, scheme=scheme, name=name, n_splits=5, seed=SEED, save=True)

        elapsed = time.perf_counter() - start

        hgb_results[(name, scheme)] = (folds, preds)

        print(f"Completed in {elapsed / 60:.2f} minutes")
        print(f"Mean RMSE: {folds['rmse'].mean():.3f}")
        print(f"Mean MAE: {folds['mae'].mean():.3f}")


Running hgb — random
Completed in 3.32 minutes
Mean RMSE: 62.373
Mean MAE: 41.032

Running hgb — blocked
Completed in 2.47 minutes
Mean RMSE: 112.406
Mean MAE: 82.366

Running hgb — chrono
Completed in 2.48 minutes
Mean RMSE: 119.419
Mean MAE: 86.267

Running hgb_poisson — random
Completed in 3.06 minutes
Mean RMSE: 60.256
Mean MAE: 38.734

Running hgb_poisson — blocked
Completed in 2.61 minutes
Mean RMSE: 113.546
Mean MAE: 81.458

Running hgb_poisson — chrono
Completed in 2.46 minutes
Mean RMSE: 120.308
Mean MAE: 86.002
